# 007 · Dilution model (not started)

**The question.** Will this company issue many new shares soon?

**The answer.** `dilution`: 1 if shares outstanding (split-adjusted, as first filed) rise by 10% or more within 182 days, else 0. About 10% of rows are 1.

**The baseline to beat.** the base rate: the same chance (about 10%) for every row.

**Measures.** AUC, log loss, Brier score (always saying the base rate scores 0.0898), and lift in the top tenth.

**Status.** Empty on purpose: headings and instructions only. Fill it in feature 007
([docs/plan.md](../docs/plan.md), issue in the roadmap).

### Rules for every model notebook

- **Spec first.** Before filling this notebook, run `/speckit-specify` for this feature with
  `SPECIFY_FEATURE_DIRECTORY=specs/007-dilution-model` on branch `007-dilution-model` (README, "Working
  together"). The notebook is for trying ideas; the chosen model goes into
  `src/sentinel/models/` and runs by command, because `sentinel update` scores every company each
  night without a notebook (docs/plan.md, "Notebooks and code").
- **Data.** Read only `data/exports/ml/dilution.parquet` through `sentinel.eda.load("dilution")`, which keeps
  the development rows (dated 2022 or earlier). Learn and test only on rows with
  `eligible = true` and an answer. Never read `data/exports/reference/` (answers for 2023 on) and
  never compute anything for 2023 or later: that is the sealed holdout, tested once by feature 010
  (constitution IV).
- **Never use as features:** `dilution`, `window_end`, `cik`, `as_of` and the flags.
- **Evaluation is fixed** (decision D60, `config/risk_params.yaml` v5, never edited): yearly
  walk-forward test years 2019, 2020, 2021, 2022; for each, train only on rows whose answer window
  ended before 1 January of that year (`sentinel.eda.purge_table` counts them); four levels:
  naive baseline → rule → classic model → LightGBM; choose the **simplest level within one
  standard error of the best**; report every test year, not only the average.
- **Show forecasts as a chance in five bands, each with its real dilution rate on the test years** with their hit rates on the test years (D62), never as a bare
  number.
- **Reproducible:** sort rows by `cik`, `as_of` before modelling; fixed seeds (`SEED = 0`); log
  each run in MLflow with the git commit and the export date. Two runs must give identical
  predictions.
- **Private data:** outputs are stripped on commit (nbstripout); written conclusions quote
  aggregates only, never one company's prices or returns (Tiingo terms). Never write to `data/`.
- **Run it** start to finish under the memory cap:
  `systemd-run --user --scope -p MemoryMax=5G -p MemorySwapMax=0 uv run jupyter nbconvert --to notebook --execute --inplace notebooks/007-dilution-model.ipynb`

### What the EDA found for this model (notebooks/004-eda-dilution.ipynb, Findings)

- **Rule level ready-made:** "an offering filed in the last 12 months" (`n_offering_12m > 0`):
  AUC 0.70–0.73 in every test year; dilution 21.8% if flagged, 3.8% if not. A model must clearly
  beat it.
- Strongest features: offerings and shelf filings, share growth over 12 months, interest
  coverage (low or missing), R&D intensity, stock-based pay to revenue, volatility, low price,
  small size, loss-making. Macro and short volume add nothing (drop them from the classic
  model).
- Missing values carry meaning (no interest coverage often means a small company without debt):
  add "was missing" flags.
- 77% of eligible foreign-filer rows have no answer: open question V2-27.
- Rows with a stock split in the window dilute 25% vs 9.7%: open question V2-25 (named cases
  first). Impossible share counts (V2-23) can make wrong answers.

Data fixes proposed but **not yet applied** (docs/decisions.md, V2-22 to V2-24: foreign filers'
market value, impossible share counts, insider dollar typos): until the researcher decides and the
pipeline is fixed, cut extreme values at the 1st and 99th percentiles of the training rows and
note it.

## 1. Setup and data

*To do:* Load the development rows, keep eligible rows with an answer, list the features to use (the EDA's advice below), and print rows and companies per test year.

## 2. Walk-forward folds

*To do:* Build the four folds (test years 2019–2022) with the purge; print train and test rows per fold.

## 3. Level 0: the naive baseline

*To do:* Score the baseline on each test year with the chosen measures.

## 4. Level 1: a rule

*To do:* A simple, explainable rule (see the EDA's suggestion below); fit only on each fold's training rows.

## 5. Level 2: a classic model

*To do:* A linear or logistic model: one feature of each redundant pair, ratios cut at their 1st and 99th percentiles (on training rows), missing values filled with the training median plus a 'was missing' flag, standardised.

## 6. Level 3: LightGBM

*To do:* Gradient-boosted trees with fixed seeds and settings chosen by time-ordered validation inside the training years only (never on the test year).

## 7. Comparison and choice

*To do:* One table: each level × each test year, the mean and its standard error; pick the simplest level within one standard error of the best.

## 8. Calibration and bands

*To do:* Calibrate the chances on earlier years (for example isotonic or Platt on the training years' last part); five bands by predicted chance, each with its real dilution rate per test year (D47).

## 9. What the model uses

*To do:* Feature importance (LightGBM) or coefficients (classic); check that the strongest features make sense and none is a leak.

## 10. Findings

*To do:* In plain words: the chosen level and why, its skill per test year against the baseline, the bands and their hit rates, what to move into `src/sentinel/models/`, and open questions.